In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from VideoGenerator import VideoGenerator
import concurrent.futures
import os
import shutil
import numpy as np
import cv2
from matplotlib import pyplot as plt
from matplotlib import cm
from PIL import Image
from tqdm import tqdm

In [3]:
generator = VideoGenerator(topNm = 3500, dim=144)

In [4]:
def generateVideo(i):
    images, cleanImages, maps = generator.generateVideos(1)
    return {
        'images': images[0],
        'maps': maps[0]
    }

In [5]:
totalVideos = 2000
trainVideos, valVideos, testVideos = round(totalVideos * 0.7), round(totalVideos * 0.2), round(totalVideos * 0.1)

training_data = {
    'train': {},
    'val': {},
    'test': {}
}

with concurrent.futures.ProcessPoolExecutor() as executor:
        trainResults = list(tqdm(executor.map(generateVideo, [i for i in range(trainVideos)])))
        for i, result in enumerate(trainResults):
            training_data['train'][f'pair{i}'] = result
        print('finished train')
        
        valResults = list(tqdm(executor.map(generateVideo, [i for i in range(valVideos)])))
        for i, result in enumerate(valResults):
            training_data['val'][f'pair{i}'] = result
        print('finished val')
        
        testResults = list(tqdm(executor.map(generateVideo, [i for i in range(testVideos)])))
        for i, result in enumerate(testResults):
            training_data['test'][f'pair{i}'] = result
        print('finished test')

        
print('ALL DONE!')

272it [01:32,  2.93it/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py", line 3552, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_23267/2894618728.py", line 11, in <module>
    trainResults = list(tqdm(executor.map(generateVideo, [i for i in range(trainVideos)])))
  File "/opt/conda/lib/python3.7/site-packages/tqdm/std.py", line 1195, in __iter__
    for obj in iterable:
  File "/opt/conda/lib/python3.7/concurrent/futures/process.py", line 483, in _chain_from_iterable_of_lists
    for element in iterable:
  File "/opt/conda/lib/python3.7/concurrent/futures/_base.py", line 598, in result_iterator
    yield fs.pop().result()
  File "/opt/conda/lib/python3.7/concurrent/futures/_base.py", line 430, in result
    self._condition.wait(timeout)
  File "/opt/conda/lib/python3.7/threading.py", line 296, in wait
    waiter.acquire()
KeyboardInterrupt

During handling of the above exception, another exception o

TypeError: object of type 'NoneType' has no len()

In [ ]:
def extraction(struct, dim = 144, frames = 12):
    keys = struct.keys()
    num_data = len(keys)
    
    images = np.zeros((num_data, frames, dim, dim, 3))
    maps = np.zeros((num_data, frames, dim, dim))
    
    i = 0
    
    for key in keys:
        res = struct[key]
        
        
        images[i] = res['images']
        maps[i] = res['maps']
    
        i = i+1
        
    return images, maps
        

In [ ]:
ti, tm = extraction(training_data['train'])
vi, vm = extraction(training_data.val)
ei, em = extraction(training_data.test)

In [ ]:
np.save("ti.npy", ti)
np.save("tm.npy", tm)
np.save("vi.npy", vi)
np.save("vm.npy", vm)
np.save("ei.npy", ei)
np.save("em.npy", em)
